# Task 7: ResNet-50 style bottleneck block + grouped/depthwise conv

In [1]:
import torch
import torch.nn as nn


In [2]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return torch.relu(self.bn(x))


In [3]:
class Bottleneck(nn.Module):
    def __init__(self, in_ch, mid_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, mid_ch, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.conv2 = DepthwiseSeparableConv(mid_ch, mid_ch, stride)
        self.conv3 = nn.Conv2d(mid_ch, out_ch, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.skip = None
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        identity = x if self.skip is None else self.skip(x)
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.conv2(out)
        out = self.bn3(self.conv3(out))
        return torch.relu(out + identity)


In [4]:
model = Bottleneck(64, 32, 128, stride=2)
x = torch.randn(1, 64, 56, 56)
print(model(x).shape)
print("params:", sum(p.numel() for p in model.parameters()))


torch.Size([1, 128, 28, 28])
params: 16288


In [5]:
from torch.profiler import profile, ProfilerActivity

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    model(x)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=5))


--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                    aten::conv2d         1.11%     100.200us        65.39%       5.895ms       1.179ms             5  
               aten::convolution         2.59%     233.800us        64.28%       5.795ms       1.159ms             5  
              aten::_convolution         1.31%     118.000us        61.68%       5.561ms       1.112ms             5  
        aten::mkldnn_convolution        58.29%       5.255ms        60.37%       5.443ms       1.089ms             5  
                aten::batch_norm         3.87%     349.300us        26.42%       2.381ms     595.375us             4  
--------------------------------  ------------  